<h1 style="text-align: center;">Project Title: AI Driven Risk Intelligence for 2026 Interpol Fugitives</h1>


### Course and Section: CS610 Applied Machine Learning (G1)

## Pillar 1

## Import libraries and packages

In [1]:
import os
import warnings
import json
import ast
from io import StringIO
from datetime import datetime
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import networkx as nx
import requests

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD, LatentDirichletAllocation
from sklearn.preprocessing import normalize, StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.pipeline import make_pipeline
from sklearn.cluster import KMeans, DBSCAN
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    silhouette_score, silhouette_samples, davies_bouldin_score,
    calinski_harabasz_score, roc_auc_score, precision_score,
    recall_score, f1_score, classification_report
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier, IsolationForest
)
from sklearn.svm import SVC

from sentence_transformers import SentenceTransformer

import pickle

warnings.filterwarnings('ignore')

# Jupyter detection
def _in_jupyter():
    try:
        shell = get_ipython().__class__.__name__
        return shell in ('ZMQInteractiveShell', 'TerminalInteractiveShell')
    except NameError:
        return False

IN_JUPYTER = _in_jupyter()
if not IN_JUPYTER:
    matplotlib.use('Agg')

if IN_JUPYTER:
    get_ipython().run_line_magic('matplotlib', 'inline')
    plt.rcParams['figure.dpi'] = 120

C:\Users\dreyw\anaconda3\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


### Loading

In [2]:
df_ready = pd.read_csv('crime_analysis_results_aft_transformer_ner.csv')

## Performance Evaluation

Identity Resolution

Generates synthetic positive pairs and negative pairs, then measures MAP, Recall, and cosine similarity separation between positives and negatives.                                                                                             

What performance evaluation is for (before weights) <br>
Each pillar is evaluated on its own appropriate metric to prove it works as a standalone component. We are not comparing pillars against each other but validating that each one does its job:<br>

- Identity Resolution → does it retrieve the right fugitive? (Recall, MAP)

In [ ]:
"""
=============================================================================
CS610 INTERPOL PROJECT — COMPONENT PERFORMANCE EVALUATION
=============================================================================
Evaluates component INDEPENDENTLY before weighting:
  1. Identity Resolution  — TF-IDF char n-gram vs SBERT name matching

Run AFTER interpol_full_pipeline.ipynb (needs outputs/interpol_final_fugitive_db.csv).
=============================================================================
"""

# ─────────────────────────────────────────────────────────────────────────────
# LOAD DATA
# ─────────────────────────────────────────────────────────────────────────────
DB_PATH = os.path.join(os.getcwd(), 'outputs', 'interpol_final_fugitive_db.csv')
db = pd.read_csv(DB_PATH).reset_index(drop=True)
db['sanctions_clean']     = db['sanctions_clean'].fillna('unknown offense')
db['name']                = db['name'].fillna('UNKNOWN')
db['detected_crime_type'] = db['detected_crime_type'].fillna('Unknown')

print(f"✅  Loaded {len(db):,} fugitives")
print(f"   Crime types: {db['detected_crime_type'].unique()}\n")

DARK_BG  = '#040810'
PANEL_BG = '#06090f'
GRID_CLR = '#1e2535'
TEXT_CLR = '#94a3b8'

def save_fig(fig, fname):
    fig.savefig(fname, dpi=150, bbox_inches='tight', facecolor=DARK_BG)
    plt.close(fig)
    print(f"   📊 Saved: {fname}")

# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 1 — IDENTITY RESOLUTION PERFORMANCE
# ═══════════════════════════════════════════════════════════════════════════════
print("=" * 60)
print("  SECTION 1: IDENTITY RESOLUTION PERFORMANCE")
print("=" * 60)

# ── Shared test pairs (used by BOTH TF-IDF and SBERT) ─────────────────────────
def perturb_name(name):
    parts = str(name).strip().split()
    variants = []
    if len(parts) >= 2:
        variants.append(f"{parts[0][0]}. {' '.join(parts[1:])}")  # A. Rahman
        variants.append(' '.join(reversed(parts)))                  # Rahman Abdul
    if len(parts) >= 3:
        variants.append(f"{parts[0]} {parts[-1]}")                 # drop middle
    variants.append(name.upper())
    variants.append(name.lower())
    return variants

rng = np.random.default_rng(42)
multi_word_pos = np.where(db['name'].str.split().str.len() >= 2)[0]
sampled_pos    = rng.choice(multi_word_pos, size=min(500, len(multi_word_pos)), replace=False)

positive_pairs = []
for pos in sampled_pos:
    orig = db.iloc[int(pos)]['name']
    for pname in perturb_name(orig)[:2]:
        positive_pairs.append((pname, int(pos), orig))

negative_pairs = []
sampled_list = sampled_pos.tolist()
for pname, true_pos, orig in positive_pairs[:300]:
    neg_candidates = [p for p in sampled_list if p != true_pos]
    neg_pos = int(rng.choice(neg_candidates))
    negative_pairs.append((pname, neg_pos, db.iloc[neg_pos]['name']))

print(f"   Positive test pairs : {len(positive_pairs)}")
print(f"   Negative test pairs : {len(negative_pairs)}")

K_VALUES = [1, 3, 5, 10]

# ─────────────────────────────────────────────────────────────────────────────
# 1a. TF-IDF CHAR N-GRAM EVALUATION
# ─────────────────────────────────────────────────────────────────────────────
print("\n── 1a. TF-IDF Char N-gram ──────────────────────────────────")

name_vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(2,4),
                            max_features=5000, sublinear_tf=True)
name_mat = name_vec.fit_transform(db['name'].str.upper().astype(str))
name_emb = normalize(name_mat)

def query_tfidf(q):
    qv = normalize(name_vec.transform([q.upper()]))
    return cosine_similarity(qv, name_emb)[0]

tfidf_recall_at_k = {k: [] for k in K_VALUES}
tfidf_ap_scores   = []
tfidf_pos_sims    = []
tfidf_neg_sims    = []

for pname, true_pos, orig in positive_pairs:
    sims   = query_tfidf(pname)
    ranked = np.argsort(sims)[::-1]
    for k in K_VALUES:
        tfidf_recall_at_k[k].append(1 if true_pos in ranked[:k] else 0)
    hits, psum = 0, 0.0
    for rank, pos in enumerate(ranked[:10], 1):
        if pos == true_pos:
            hits += 1
            psum += hits / rank
    tfidf_ap_scores.append(psum / max(hits, 1))
    tfidf_pos_sims.append(float(sims[true_pos]))

for pname, neg_pos, _ in negative_pairs:
    sims = query_tfidf(pname)
    tfidf_neg_sims.append(float(sims[neg_pos]))

tfidf_map = float(np.mean(tfidf_ap_scores))
print(f"   MAP                      : {tfidf_map:.4f}")
for k in K_VALUES:
    r = np.mean(tfidf_recall_at_k[k])
    print(f"   Recall@{k:<2}               : {r:.4f}  ({r*100:.1f}%)")
print(f"   Mean cosine — positives  : {np.mean(tfidf_pos_sims):.4f}")
print(f"   Mean cosine — negatives  : {np.mean(tfidf_neg_sims):.4f}")
print(f"   Separation (pos − neg)   : {np.mean(tfidf_pos_sims) - np.mean(tfidf_neg_sims):.4f}")

# ─────────────────────────────────────────────────────────────────────────────
# 1b. SBERT EVALUATION
# ─────────────────────────────────────────────────────────────────────────────
print("\n── 1b. SBERT (all-MiniLM-L6-v2) ───────────────────────────")

sbert_model = SentenceTransformer('all-MiniLM-L6-v2')
sbert_emb   = normalize(sbert_model.encode(
                  db['name'].str.upper().tolist(),
                  batch_size=128, show_progress_bar=True))

def query_sbert(q):
    qv = normalize(sbert_model.encode([q.upper()]))
    return cosine_similarity(qv, sbert_emb)[0]

sbert_recall_at_k = {k: [] for k in K_VALUES}
sbert_ap_scores   = []
sbert_pos_sims    = []
sbert_neg_sims    = []

for pname, true_pos, orig in positive_pairs:
    sims   = query_sbert(pname)
    ranked = np.argsort(sims)[::-1]
    for k in K_VALUES:
        sbert_recall_at_k[k].append(1 if true_pos in ranked[:k] else 0)
    hits, psum = 0, 0.0
    for rank, pos in enumerate(ranked[:10], 1):
        if pos == true_pos:
            hits += 1
            psum += hits / rank
    sbert_ap_scores.append(psum / max(hits, 1))
    sbert_pos_sims.append(float(sims[true_pos]))

for pname, neg_pos, _ in negative_pairs:
    sims = query_sbert(pname)
    sbert_neg_sims.append(float(sims[neg_pos]))

sbert_map = float(np.mean(sbert_ap_scores))
print(f"   MAP                      : {sbert_map:.4f}")
for k in K_VALUES:
    r = np.mean(sbert_recall_at_k[k])
    print(f"   Recall@{k:<2}               : {r:.4f}  ({r*100:.1f}%)")
print(f"   Mean cosine — positives  : {np.mean(sbert_pos_sims):.4f}")
print(f"   Mean cosine — negatives  : {np.mean(sbert_neg_sims):.4f}")
print(f"   Separation (pos − neg)   : {np.mean(sbert_pos_sims) - np.mean(sbert_neg_sims):.4f}")

# ─────────────────────────────────────────────────────────────────────────────
# 1c. HEAD-TO-HEAD COMPARISON
# ─────────────────────────────────────────────────────────────────────────────
print("\n── 1c. TF-IDF vs SBERT — Head-to-Head ─────────────────────")
print(f"   {'Metric':<28} {'TF-IDF':>8} {'SBERT':>8} {'Winner':>8}")
print("   " + "─" * 56)

comparison_rows = [
    ("MAP",          tfidf_map,                                       sbert_map,                                       True),
    ("Recall@1",     np.mean(tfidf_recall_at_k[1]),                   np.mean(sbert_recall_at_k[1]),                   True),
    ("Recall@3",     np.mean(tfidf_recall_at_k[3]),                   np.mean(sbert_recall_at_k[3]),                   True),
    ("Recall@5",     np.mean(tfidf_recall_at_k[5]),                   np.mean(sbert_recall_at_k[5]),                   True),
    ("Mean pos sim", np.mean(tfidf_pos_sims),                         np.mean(sbert_pos_sims),                         True),
    ("Mean neg sim", np.mean(tfidf_neg_sims),                         np.mean(sbert_neg_sims),                         False),
    ("Pos−Neg sep",  np.mean(tfidf_pos_sims)-np.mean(tfidf_neg_sims), np.mean(sbert_pos_sims)-np.mean(sbert_neg_sims), True),
]

for metric, tfidf_val, sbert_val, higher_better in comparison_rows:
    if higher_better:
        winner = "TF-IDF ✓" if tfidf_val >= sbert_val else "SBERT  ✓"
    else:
        winner = "TF-IDF ✓" if tfidf_val <= sbert_val else "SBERT  ✓"
    print(f"   {metric:<28} {tfidf_val:>8.4f} {sbert_val:>8.4f} {winner:>8}")

winner_count = sum(
    1 for _, tv, sv, hb in comparison_rows
    if (hb and tv >= sv) or (not hb and tv <= sv)
)
print(f"\n   TF-IDF wins {winner_count}/{len(comparison_rows)} metrics")
print(f"   → TF-IDF char n-grams selected for Pillar 1 pipeline.")
print(f"     Character-level n-grams capture transliterations,")
print(f"     abbreviations and reversed name orders that SBERT")
print(f"     treats as semantically distant full-sentence tokens.")

# ─────────────────────────────────────────────────────────────────────────────
# 1d. PLOTS
# ─────────────────────────────────────────────────────────────────────────────
recall_at_k = tfidf_recall_at_k
ap_scores   = tfidf_ap_scores
pos_sims    = tfidf_pos_sims
neg_sims    = tfidf_neg_sims
map_score   = tfidf_map

fig = plt.figure(figsize=(18, 4), facecolor=DARK_BG)
gs  = gridspec.GridSpec(1, 4, figure=fig, wspace=0.35)

# Panel 1 — Recall@k
ax1 = fig.add_subplot(gs[0])
ax1.set_facecolor(PANEL_BG)
k_vals = [np.mean(recall_at_k[k]) for k in K_VALUES]
bars = ax1.bar([f'@{k}' for k in K_VALUES], k_vals,
               color=['#38bdf8','#34d399','#a78bfa','#f59e0b'],
               edgecolor=DARK_BG, linewidth=1.5)
ax1.set_ylim(0, 1.15)
ax1.set_title('Recall@k — TF-IDF', color='#e2e8f0', fontfamily='monospace', fontsize=10, pad=8)
ax1.set_ylabel('Recall', color=TEXT_CLR, fontsize=9)
ax1.tick_params(colors=TEXT_CLR, labelsize=9)
ax1.grid(True, color=GRID_CLR, linewidth=0.5, alpha=0.4, axis='y')
for spine in ax1.spines.values(): spine.set_edgecolor(GRID_CLR)
for bar, val in zip(bars, k_vals):
    ax1.text(bar.get_x()+bar.get_width()/2, val+0.02, f'{val:.3f}',
             ha='center', color='#e2e8f0', fontsize=9, fontfamily='monospace')
ax1.axhline(map_score, color='#ff2d2d', linestyle='--', linewidth=1.2, label=f'MAP={map_score:.3f}')
ax1.legend(fontsize=8, framealpha=0.2, labelcolor='white')

# Panel 2 — Cosine similarity distributions
ax2 = fig.add_subplot(gs[1])
ax2.set_facecolor(PANEL_BG)
ax2.hist(pos_sims, bins=40, color='#34d399', alpha=0.75, label='Positive pairs', edgecolor=DARK_BG)
ax2.hist(neg_sims, bins=40, color='#ff2d2d', alpha=0.75, label='Negative pairs', edgecolor=DARK_BG)
ax2.set_title('Cosine Sim — TF-IDF', color='#e2e8f0', fontfamily='monospace', fontsize=10, pad=8)
ax2.set_xlabel('Cosine Similarity', color=TEXT_CLR, fontsize=9)
ax2.set_ylabel('Count', color=TEXT_CLR, fontsize=9)
ax2.tick_params(colors=TEXT_CLR, labelsize=8)
ax2.legend(fontsize=8, framealpha=0.2, labelcolor='white')
ax2.grid(True, color=GRID_CLR, linewidth=0.5, alpha=0.4)
for spine in ax2.spines.values(): spine.set_edgecolor(GRID_CLR)

# Panel 3 — Average Precision distribution
ax3 = fig.add_subplot(gs[2])
ax3.set_facecolor(PANEL_BG)
ax3.hist(ap_scores, bins=30, color='#a78bfa', edgecolor=DARK_BG, alpha=0.85)
ax3.axvline(map_score, color='#ff2d2d', linestyle='--', linewidth=1.5, label=f'MAP={map_score:.3f}')
ax3.set_title('Avg Precision — TF-IDF', color='#e2e8f0', fontfamily='monospace', fontsize=10, pad=8)
ax3.set_xlabel('Average Precision', color=TEXT_CLR, fontsize=9)
ax3.set_ylabel('Count', color=TEXT_CLR, fontsize=9)
ax3.tick_params(colors=TEXT_CLR, labelsize=8)
ax3.legend(fontsize=8, framealpha=0.2, labelcolor='white')
ax3.grid(True, color=GRID_CLR, linewidth=0.5, alpha=0.4)
for spine in ax3.spines.values(): spine.set_edgecolor(GRID_CLR)

# Panel 4 — TF-IDF vs SBERT Recall@k comparison
ax4 = fig.add_subplot(gs[3])
ax4.set_facecolor(PANEL_BG)
x             = np.arange(len(K_VALUES))
w             = 0.35
tfidf_recalls = [np.mean(tfidf_recall_at_k[k]) for k in K_VALUES]
sbert_recalls = [np.mean(sbert_recall_at_k[k]) for k in K_VALUES]
ax4.bar(x - w/2, tfidf_recalls, w, color='#38bdf8', label='TF-IDF', edgecolor=DARK_BG)
ax4.bar(x + w/2, sbert_recalls, w, color='#f97316', label='SBERT',  edgecolor=DARK_BG)
ax4.set_xticks(x)
ax4.set_xticklabels([f'@{k}' for k in K_VALUES], color=TEXT_CLR, fontsize=9)
ax4.set_ylim(0, 1.15)
ax4.set_title('TF-IDF vs SBERT Recall@k', color='#e2e8f0', fontfamily='monospace', fontsize=10, pad=8)
ax4.set_ylabel('Recall', color=TEXT_CLR, fontsize=9)
ax4.tick_params(colors=TEXT_CLR, labelsize=9)
ax4.grid(True, color=GRID_CLR, linewidth=0.5, alpha=0.4, axis='y')
ax4.legend(fontsize=8, framealpha=0.2, labelcolor='white')
for spine in ax4.spines.values(): spine.set_edgecolor(GRID_CLR)
ax4.axhline(tfidf_map, color='#38bdf8', linestyle='--', linewidth=1, alpha=0.6)
ax4.axhline(sbert_map, color='#f97316', linestyle='--', linewidth=1, alpha=0.6)

fig.suptitle('PILLAR 1 — Identity Resolution: TF-IDF Char N-gram (selected) vs SBERT (baseline)',
             color='#e2e8f0', fontfamily='monospace', fontsize=11, y=1.02)
save_fig(fig, 'outputs/eval_identity_resolution.png')

# ─────────────────────────────────────────────────────────────────────────────
# SUMMARY TABLE
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  PERFORMANCE SUMMARY")
print("=" * 60)

summary = pd.DataFrame({
    "Component": [
        "Identity Resolution (TF-IDF char n-gram)",
        "Identity Resolution (TF-IDF char n-gram)",
        "Identity Resolution (TF-IDF char n-gram)",
        "Identity Resolution (TF-IDF char n-gram)",
        "Identity Resolution (TF-IDF char n-gram)",
        "Identity Resolution — SBERT baseline",
        "Identity Resolution — SBERT baseline",
        "Identity Resolution — SBERT baseline",
    ],
    "Metric": [
        "MAP (Mean Avg Precision)", "Recall@1", "Recall@3",
        "Mean cosine (positives)", "Mean cosine (negatives)",
        "MAP (Mean Avg Precision)", "Recall@1", "Recall@3",
    ],
    "Value": [
        f"{tfidf_map:.4f}",
        f"{np.mean(tfidf_recall_at_k[1]):.4f}",
        f"{np.mean(tfidf_recall_at_k[3]):.4f}",
        f"{np.mean(tfidf_pos_sims):.4f}",
        f"{np.mean(tfidf_neg_sims):.4f}",
        f"{sbert_map:.4f}",
        f"{np.mean(sbert_recall_at_k[1]):.4f}",
        f"{np.mean(sbert_recall_at_k[3]):.4f}",
    ],
    "Direction": [
        "↑ higher better", "↑", "↑", "↑", "↓ lower better",
        "↑ higher better", "↑", "↑",
    ]
})

print(summary.to_string(index=False))
summary.to_csv('outputs/performance_summary.csv', index=False)
print("\n  Saved: outputs/performance_summary.csv")
print("  Saved: outputs/eval_identity_resolution.png")
print("\nDone.")

✅  Loaded 7,057 fugitives
   Crime types: ['terrorism' 'weapons' 'homicide' 'assault' 'Unknown' 'financial crime'
 'narcotics' 'cyber crime']

  SECTION 1: IDENTITY RESOLUTION PERFORMANCE
   Positive test pairs : 1000
   Negative test pairs : 300

── 1a. TF-IDF Char N-gram ──────────────────────────────────
   MAP                      : 0.8857
   Recall@1                : 0.8160  (81.6%)
   Recall@3                : 0.9540  (95.4%)
   Recall@5                : 0.9690  (96.9%)
   Recall@10               : 0.9860  (98.6%)
   Mean cosine — positives  : 0.8754
   Mean cosine — negatives  : 0.0213
   Separation (pos − neg)   : 0.8541

── 1b. SBERT (all-MiniLM-L6-v2) ───────────────────────────


Batches:   0%|          | 0/56 [00:00<?, ?it/s]

- Identity Resolution results: MAP of 0.886 and Recall@3 of 0.954 means that 95% of the time, the correct fugitive appears in the top 3 results when a perturbed name variant is submitted.

# Description and Settings for Pillar 1

The weights are not about which pillar performs "better" — they're about how much each pillar should contribute to the final risk score given its role in the pipeline. TF-IDF Char N-gram gets 40% not because it outperforms crime severity, but because name identity is the primary screening signal in AML. Crime severity gets 30% because it captures danger level. The weights reflect domain logic, not a performance race between pillars.

In [ ]:
BASE_DIR   = os.getcwd()
OUTPUT_DIR = os.path.join(BASE_DIR, "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

DARK_BG  = '#040810'
PANEL_BG = '#06090f'
GRID_CLR = '#1e2535'
TEXT_CLR = '#94a3b8'

## PILLAR 1

In [ ]:
# =============================================================================
# PILLAR 1: TF-IDF CHAR N-GRAM IDENTITY RESOLUTION
# =============================================================================
print("\n" + "─" * 65)
print("  PILLAR 1 — TF-IDF CHAR N-GRAM: IDENTITY RESOLUTION  (weight: 40%)")
print("─" * 65)

def build_name_embeddings(names: pd.Series):
    """
    Char-level TF-IDF (2-4 gram) + L2 normalisation.
    Character n-grams chosen over sentence embeddings (SBERT) as they
    provide superior sub-word sensitivity for AML name screening —
    handling transliterations, abbreviations, and reversed name orders.
    """
    vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4),
                          max_features=5000, sublinear_tf=True)
    mat = vec.fit_transform(names.str.upper().astype(str))
    return normalize(mat), vec

def query_tfidf(client_name: str, vectorizer, db_embeddings) -> np.ndarray:
    """
    Return cosine similarity array of shape (n_fugitives,).
    Called at inference time — pass result as tfidf_name_score into Pillar 3.
    """
    q = normalize(vectorizer.transform([client_name.upper()]))
    return cosine_similarity(q, db_embeddings)[0]

print("\n⚙  Building name embeddings...")
name_embeddings, name_vectorizer = build_name_embeddings(df_ready['name'])
print(f"   Shape: {name_embeddings.shape}")

# ── Missing value report before modelling ─────────────────────────────────────
print("\n   ── Pre-modelling null check ──────────────────────────────")
key_cols = ['name', 'GENDER', 'age_today', 'height', 'eyeColor', 'hairColor',
            'sanctions_clean', 'detected_crime_type']
for col in key_cols:
    if col in df_ready.columns:
        n_null = df_ready[col].isnull().sum()
        pct    = n_null / len(df_ready) * 100
        flag   = "  ⚠️" if pct > 10 else ""
        print(f"   {col:<22} null: {n_null:>5} ({pct:>5.1f}%){flag}")
print("   ─────────────────────────────────────────────────────────")

# ── Demo queries ──────────────────────────────────────────────────────────────
QUERIES = [
    # Existing queries
    ("S. Nikitenko",     "Initial + surname"),
    ("Hasen Aksema",     "Missing middle name"),
    ("Abdul Sambolotov", "Single char typo"),
    ("JIAN XIA",         "Exact match"),
    ("Norbert Bialas",   "Case mismatch"),
    ("Ramazan Chigayev", "Transliteration variant"),
    ("M. Al-Saeed",      "Abbreviated first name"),
    ("Levitan Elyasov",  "Cross-script: и→i, я→ya"),

    # ── NEW: Real INTERPOL fugitive name variants ──────────────────
    # Original: JOSE ANTONIO SOLARES GONZALEZ
    ("Jose Solares Gonzalez",    "Missing middle name"),
    ("J. Antonio Solares",       "Initial + dropped surname"),
    ("Gonzalez Jose Antonio",    "Reversed name order"),

    # Original: SHYAMAL RAO REDDY
    ("Shyamal Rao",              "Missing surname"),
    ("S. Rao Reddy",             "Initial + surname"),
    ("Shyamal Roa Reddy",        "Typo in middle name"),

    # Original: MARLON JAMES WINTERS
    ("M. James Winters",         "Initial + surname"),
    ("Marlon Winters",           "Missing middle name"),
    ("Marlon James Wynters",     "Typo in surname"),
]

p1_results = []
print(f"\n  {'Client Name':<26} {'Top Match':<28} {'Score':>7}  Flag")
print("  " + "─" * 72)

for client_name, desc in QUERIES:
    sims     = query_tfidf(client_name, name_vectorizer, name_embeddings)
    top_idx  = int(np.argmax(sims))
    top_name = df_ready.iloc[top_idx]['name']
    top_sc   = float(sims[top_idx])
    top3     = [(df_ready.iloc[i]['name'], round(float(sims[i]), 4))
                for i in np.argsort(sims)[::-1][:3]]
    flag = ("🔴 HIGH CONF" if top_sc >= 0.75
            else "🟡 REVIEW"  if top_sc >= 0.50
            else "✅ LOW")
    p1_results.append(dict(client_name=client_name, description=desc,
                           top_match=top_name, tfidf_name_score=top_sc,
                           top3=top3, top_idx=top_idx, flag=flag))
    print(f"  {client_name:<26} {top_name:<28} {top_sc:>7.4f}  {flag}  ({desc})")

print(f"\n  Thresholds (Pillar 1, 40% weight):")
print(f"  ≥ 0.90  Critical  | 0.75–0.90  High | 0.50–0.75  Review | <0.50  Low")

# ── Top-3 detail for each query ───────────────────────────────────────────────
print(f"\n  Top-3 Candidates per Query:")
print(f"  {'Client':<26} {'Rank':<5} {'Match':<28} {'Score':>7}")
print("  " + "─" * 70)
for r in p1_results:
    for rank, (name, sc) in enumerate(r['top3'], 1):
        color_flag = "🔴" if sc >= 0.75 else "🟡" if sc >= 0.50 else "  "
        if rank == 1:
            print(f"  {r['client_name']:<26} #{rank:<4} {name:<28} {sc:>7.4f} {color_flag}  ({r['description']})")
        else:
            print(f"  {'':26} #{rank:<4} {name:<28} {sc:>7.4f} {color_flag}")
    print("  " + "·" * 70)

## Pickling

In [ ]:
BASE_DIR   = os.getcwd()
OUTPUT_DIR = os.path.join(BASE_DIR, "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

with open(os.path.join(OUTPUT_DIR, 'pillar1_name_vectorizer.pkl'), 'wb') as f:
    pickle.dump(name_vectorizer, f)

with open(os.path.join(OUTPUT_DIR, 'pillar1_name_embeddings.pkl'), 'wb') as f:
    pickle.dump(name_embeddings, f)

print("  Pillar 1 pickled:")
print(f"   → {os.path.join(OUTPUT_DIR, 'pillar1_name_vectorizer.pkl')}")
print(f"   → {os.path.join(OUTPUT_DIR, 'pillar1_name_embeddings.pkl')}")

In [ ]:
with open('outputs/pillar1_name_vectorizer.pkl', 'rb') as f:
    name_vectorizer = pickle.load(f)

with open('outputs/pillar1_name_embeddings.pkl', 'rb') as f:
    name_embeddings = pickle.load(f)

def query_tfidf(q):
    qv = normalize(name_vectorizer.transform([q.upper()]))
    return cosine_similarity(qv, name_embeddings)[0]

# Test it
sims = query_tfidf("S. Nikitenko")
print("  Loaded successfully")